# 🧪 Exploration des données (Couche Bronze)

Ce notebook permet de visualiser et d'explorer les données ingérées dans la couche Bronze. 
**Objectif :** Vérifier la qualité des données, identifier les anomalies (valeurs nulles, doublons) avant de passer à la couche Silver.

In [ ]:
import os
from src.common.spark_session_manager import get_spark_session
from src.config import PRESIDENTIELLE_BRONZE_PATH
from pyspark.sql.functions import col

# 1. Initialiser la session Spark
spark = get_spark_session(app_name="Exploration_Bronze")

# 2. Configuration "Look Databricks" (Eager Evaluation)
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)

## 📊 Exploration de l'Élection Présidentielle 2022 (Tour 1)
Lecture des données depuis le format **Parquet** (stockage optimisé).

In [ ]:
t1_path = os.path.join(PRESIDENTIELLE_BRONZE_PATH, "lyon_T1_presidentiel_2022")
df_t1 = spark.read.parquet(t1_path)

# Aperçu des données
df_t1

## 🔬 Analyse Technique des Schémas (Data Profiling)
Il est crucial de vérifier les types détectés par Spark pour anticiper le nettoyage en couche Silver.

In [ ]:
# Affichage des types de colonnes (Senior Approach)
print("--- Analyse des types de données ---")
for col_name, dtype in df_t1.dtypes:
    if "string" in dtype:
        print(f"⚠️ Colonne TEXTE (potentiel nettoyage requis) : {col_name}")
    else:
        print(f"✅ Colonne NUMÉRIQUE : {col_name}")

## 🔍 Statistiques Descriptives
On se concentre sur les colonnes de participation pour vérifier la cohérence des chiffres (Inscrits, Votants, Exprimés).

In [ ]:
numeric_cols = ["Inscrits", "Abstentions", "Votants", "Blancs", "Nuls", "Exprimés"]
df_t1.select(numeric_cols).describe().toPandas()

## 🌍 Validation du Périmètre Géographique
On s'assure que notre dataset ne contient QUE la ville de Lyon (Code INSEE 69123).

In [ ]:
df_t1.groupBy("Code de la commune", "Libellé de la commune").count()